# Basic NLP Tasks with the OpenAI Chat Completions API — Starter Notebook

In this segment, you will use OpenAI's **Chat Completions API** to perform basic NLP tasks:

1. **MCQ 1 – Text classification:** tag each laptop with a user category (`gamer`, `business`, `general`, `programmer`).
2. **MCQ 2 – Information extraction:** pull structured laptop tags (brand, GPU, weight, budget, ...) out of free-text descriptions as JSON.
3. **MCQ 3 – Question answering with Wikipedia:** search Wikipedia for context and let the model answer from it.

Complete the cells marked **1.1–1.3**, **2.1–2.3** and **3.1–3.3** (replace every `___`).

**Note:** The library setup and the data are already included. Just run the setup cells.

## 0. Setup

In [ ]:
!pip install -q --upgrade openai pandas requests

In [ ]:
import os, json, getpass
import pandas as pd
import requests
from openai import OpenAI

# Store your key as a Colab secret named OPENAI_API_KEY (key icon in the left sidebar),
# or paste it when prompted.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()

# Change this if your account uses a different chat model (e.g. "gpt-4o-mini").
MODEL = "gpt-5.4-mini"

In [ ]:
def get_chat_completion(messages, model=MODEL, json_mode=False):
    """Send a list of messages to the Chat Completions API and return the reply text."""
    kwargs = {"model": model, "messages": messages}
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content.strip()

# quick test
print(get_chat_completion([{"role": "user", "content": "Say hello in one word."}]))

### Load the data
The laptop dataset is embedded below and written to `laptop_data.csv`.

In [ ]:
LAPTOP_CSV = r"""
Brand,Model Name,Core,CPU Manufacturer,Clock Speed,RAM Size,Storage Type,Display Type,Display Size,Graphics Processor,Screen Resolution,OS,Laptop Weight,Special Features,Warranty,Average Battery Life,Price,Description
Dell,Inspiron,Core i5,Intel,2.4 GHz,8GB,SSD,LCD,15.6 inch,Intel UHD,1920x1080,Windows 10,2.5 kg,Backlit Keyboard,1 year,6 hours,35000,"The Dell Inspiron is powered by an Intel Core i5 processor running at 2.4 GHz, paired with 8GB of RAM and SSD storage. It has a 15.6 inch LCD display with a resolution of 1920x1080 and uses Intel UHD graphics. It runs Windows 10, weighs 2.5 kg, and offers Backlit Keyboard, a warranty of 1 year and around 6 hours of battery life. It is priced at Rs. 35,000. A reliable everyday laptop for browsing, streaming, online classes and light office work."
MSI,GL65,Core i7,Intel,2.6 GHz,16GB,SSD,IPS,15.6 inch,NVIDIA GTX,1920x1080,Windows 10,2.3 kg,RGB Keyboard,2 years,4 hours,55000,"The MSI GL65 is powered by an Intel Core i7 processor running at 2.6 GHz, paired with 16GB of RAM and SSD storage. It has a 15.6 inch IPS display with a resolution of 1920x1080 and uses NVIDIA GTX graphics. It runs Windows 10, weighs 2.3 kg, and offers RGB Keyboard, a warranty of 2 years and around 4 hours of battery life. It is priced at Rs. 55,000. Built for gamers: the dedicated NVIDIA GTX graphics, high-refresh IPS panel and RGB keyboard deliver smooth frame rates in modern AAA titles."
HP,EliteBook,Core i7,Intel,2.8 GHz,16GB,SSD,LED,14 inch,Intel UHD,1920x1080,Windows 11,1.5 kg,Fingerprint Sensor,3 years,8 hours,90000,"The HP EliteBook is powered by an Intel Core i7 processor running at 2.8 GHz, paired with 16GB of RAM and SSD storage. It has a 14 inch LED display with a resolution of 1920x1080 and uses Intel UHD graphics. It runs Windows 11, weighs 1.5 kg, and offers Fingerprint Sensor, a warranty of 3 years and around 8 hours of battery life. It is priced at Rs. 90,000. A premium business laptop with enterprise-grade security, a fingerprint sensor and long battery life for professionals who travel to client meetings."
Lenovo,IdeaPad,Core i3,Intel,2.1 GHz,8GB,HDD,TN,15.6 inch,Intel UHD,1366x768,Windows 10,2.2 kg,Dolby Audio,1 year,5 hours,25000,"The Lenovo IdeaPad is powered by an Intel Core i3 processor running at 2.1 GHz, paired with 8GB of RAM and HDD storage. It has a 15.6 inch TN display with a resolution of 1366x768 and uses Intel UHD graphics. It runs Windows 10, weighs 2.2 kg, and offers Dolby Audio, a warranty of 1 year and around 5 hours of battery life. It is priced at Rs. 25,000. An affordable laptop for students and families who need a machine for everyday tasks like web browsing, documents and video calls."
ASUS,ZenBook Pro,Core i9,Intel,3.1 GHz,32GB,SSD,OLED,15.6 inch,NVIDIA RTX,3840x2160,Windows 11,1.8 kg,Touchscreen,2 years,7 hours,200000,"The ASUS ZenBook Pro is powered by an Intel Core i9 processor running at 3.1 GHz, paired with 32GB of RAM and SSD storage. It has a 15.6 inch OLED display with a resolution of 3840x2160 and uses NVIDIA RTX graphics. It runs Windows 11, weighs 1.8 kg, and offers Touchscreen, a warranty of 2 years and around 7 hours of battery life. It is priced at Rs. 200,000. Aimed at software developers and data scientists: 32GB RAM and a Core i9 handle large codebases, Docker containers, virtual machines and ML model training with ease."
Acer,Swift 3,Ryzen 5,AMD,2.1 GHz,8GB,SSD,IPS,14 inch,AMD Radeon,1920x1080,Windows 10,1.2 kg,Backlit Keyboard,1 year,10 hours,45000,"The Acer Swift 3 is powered by an AMD Ryzen 5 processor running at 2.1 GHz, paired with 8GB of RAM and SSD storage. It has a 14 inch IPS display with a resolution of 1920x1080 and uses AMD Radeon graphics. It runs Windows 10, weighs 1.2 kg, and offers Backlit Keyboard, a warranty of 1 year and around 10 hours of battery life. It is priced at Rs. 45,000. A thin and light everyday laptop for students and home users who want good battery life for notes, streaming and browsing."
Microsoft,Surface Laptop,Core i5,Intel,1.6 GHz,8GB,SSD,PixelSense,13.5 inch,Intel Iris Xe,2256x1504,Windows 11,1.3 kg,Touchscreen,1 year,11 hours,80000,"The Microsoft Surface Laptop is powered by an Intel Core i5 processor running at 1.6 GHz, paired with 8GB of RAM and SSD storage. It has a 13.5 inch PixelSense display with a resolution of 2256x1504 and uses Intel Iris Xe graphics. It runs Windows 11, weighs 1.3 kg, and offers Touchscreen, a warranty of 1 year and around 11 hours of battery life. It is priced at Rs. 80,000. A sleek business laptop for executives and consultants, with a sharp touchscreen and all-day battery for presentations and productivity apps."
Lenovo,ThinkPad,Core i7,Intel,2.7 GHz,16GB,SSD,IPS,14 inch,Intel UHD,1920x1080,Windows 11,1.6 kg,Spill-resistant Keyboard,3 years,9 hours,95000,"The Lenovo ThinkPad is powered by an Intel Core i7 processor running at 2.7 GHz, paired with 16GB of RAM and SSD storage. It has a 14 inch IPS display with a resolution of 1920x1080 and uses Intel UHD graphics. It runs Windows 11, weighs 1.6 kg, and offers Spill-resistant Keyboard, a warranty of 3 years and around 9 hours of battery life. It is priced at Rs. 95,000. The classic business workhorse: durable build, spill-resistant keyboard, TPM security and docking support for corporate environments."
Apple,MacBook Pro,M2,Apple,3.5 GHz,16GB,SSD,Retina,14 inch,Apple integrated,3024x1964,macOS,1.6 kg,Touch ID,1 year,18 hours,200000,"The Apple MacBook Pro is powered by an Apple M2 processor running at 3.5 GHz, paired with 16GB of RAM and SSD storage. It has a 14 inch Retina display with a resolution of 3024x1964 and uses Apple integrated graphics. It runs macOS, weighs 1.6 kg, and offers Touch ID, a warranty of 1 year and around 18 hours of battery life. It is priced at Rs. 200,000. A favourite among programmers and app developers: a Unix-based OS, powerful M2 chip and long battery make it ideal for coding, compiling and running development tools."
Dell,XPS,Core i7,Intel,2.8 GHz,16GB,SSD,InfinityEdge,13.4 inch,Intel Iris Xe,1920x1200,Windows 11,1.2 kg,Fingerprint Sensor,1 year,12 hours,110000,"The Dell XPS is powered by an Intel Core i7 processor running at 2.8 GHz, paired with 16GB of RAM and SSD storage. It has a 13.4 inch InfinityEdge display with a resolution of 1920x1200 and uses Intel Iris Xe graphics. It runs Windows 11, weighs 1.2 kg, and offers Fingerprint Sensor, a warranty of 1 year and around 12 hours of battery life. It is priced at Rs. 110,000. A compact premium laptop for developers who code on the go, with enough power for IDEs, local servers and multiple browser tabs."
ASUS,ROG Strix G,Core i7,Intel,2.9 GHz,16GB,SSD,IPS,15.6 inch,NVIDIA RTX,1920x1080,Windows 10,2.9 kg,RGB Keyboard,2 years,6 hours,85000,"The ASUS ROG Strix G is powered by an Intel Core i7 processor running at 2.9 GHz, paired with 16GB of RAM and SSD storage. It has a 15.6 inch IPS display with a resolution of 1920x1080 and uses NVIDIA RTX graphics. It runs Windows 10, weighs 2.9 kg, and offers RGB Keyboard, a warranty of 2 years and around 6 hours of battery life. It is priced at Rs. 85,000. A dedicated gaming laptop with NVIDIA RTX graphics, a 144Hz display and advanced cooling for competitive esports and high-end games."
Acer,Predator,Core i7,Intel,2.6 GHz,16GB,SSD,IPS,17.3 inch,NVIDIA RTX,1920x1080,Windows 11,3.0 kg,RGB Keyboard,1 year,5 hours,120000,"The Acer Predator is powered by an Intel Core i7 processor running at 2.6 GHz, paired with 16GB of RAM and SSD storage. It has a 17.3 inch IPS display with a resolution of 1920x1080 and uses NVIDIA RTX graphics. It runs Windows 11, weighs 3.0 kg, and offers RGB Keyboard, a warranty of 1 year and around 5 hours of battery life. It is priced at Rs. 120,000. A powerhouse for hardcore gamers, featuring a large 17.3-inch high-refresh display, RTX graphics and aggressive cooling for long gaming sessions."
HP,Envy x360,Ryzen 7,AMD,2.0 GHz,16GB,SSD,IPS,13.3 inch,AMD Radeon,1920x1080,Windows 11,1.3 kg,"Convertible, Touchscreen",1 year,10 hours,70000,"The HP Envy x360 is powered by an AMD Ryzen 7 processor running at 2.0 GHz, paired with 16GB of RAM and SSD storage. It has a 13.3 inch IPS display with a resolution of 1920x1080 and uses AMD Radeon graphics. It runs Windows 11, weighs 1.3 kg, and offers Convertible, Touchscreen, a warranty of 1 year and around 10 hours of battery life. It is priced at Rs. 70,000. A versatile 2-in-1 convertible for everyday users who want a laptop for work, streaming, note-taking and casual photo editing."
Lenovo,Legion,Ryzen 7,AMD,3.2 GHz,16GB,SSD,IPS,15.6 inch,NVIDIA RTX,2560x1440,Windows 11,2.4 kg,RGB Keyboard,2 years,5 hours,130000,"The Lenovo Legion is powered by an AMD Ryzen 7 processor running at 3.2 GHz, paired with 16GB of RAM and SSD storage. It has a 15.6 inch IPS display with a resolution of 2560x1440 and uses NVIDIA RTX graphics. It runs Windows 11, weighs 2.4 kg, and offers RGB Keyboard, a warranty of 2 years and around 5 hours of battery life. It is priced at Rs. 130,000. Engineered for gaming, with a QHD 165Hz screen, NVIDIA RTX graphics and a performance mode tuned for the latest titles."
Dell,Precision,Core i9,Intel,2.4 GHz,32GB,SSD,IPS,15.6 inch,NVIDIA Quadro,3840x2400,Windows 11,2.0 kg,Fingerprint Sensor,3 years,8 hours,250000,"The Dell Precision is powered by an Intel Core i9 processor running at 2.4 GHz, paired with 32GB of RAM and SSD storage. It has a 15.6 inch IPS display with a resolution of 3840x2400 and uses NVIDIA Quadro graphics. It runs Windows 11, weighs 2.0 kg, and offers Fingerprint Sensor, a warranty of 3 years and around 8 hours of battery life. It is priced at Rs. 250,000. A mobile workstation for software engineers and ML practitioners who compile large projects, run simulations and train models locally."
Acer,Aspire 5,Core i5,Intel,2.4 GHz,8GB,SSD,IPS,15.6 inch,Intel Iris Xe,1920x1080,Windows 11,1.8 kg,Backlit Keyboard,1 year,7 hours,50000,"The Acer Aspire 5 is powered by an Intel Core i5 processor running at 2.4 GHz, paired with 8GB of RAM and SSD storage. It has a 15.6 inch IPS display with a resolution of 1920x1080 and uses Intel Iris Xe graphics. It runs Windows 11, weighs 1.8 kg, and offers Backlit Keyboard, a warranty of 1 year and around 7 hours of battery life. It is priced at Rs. 50,000. A budget-friendly all-rounder for daily use: browsing, schoolwork, movies and basic office tasks."
HP,Spectre x360,Core i7,Intel,2.8 GHz,16GB,SSD,OLED,14 inch,Intel Iris Xe,3000x2000,Windows 11,1.4 kg,"Convertible, Pen support",2 years,13 hours,140000,"The HP Spectre x360 is powered by an Intel Core i7 processor running at 2.8 GHz, paired with 16GB of RAM and SSD storage. It has a 14 inch OLED display with a resolution of 3000x2000 and uses Intel Iris Xe graphics. It runs Windows 11, weighs 1.4 kg, and offers Convertible, Pen support, a warranty of 2 years and around 13 hours of battery life. It is priced at Rs. 140,000. A luxury business convertible for managers and executives, with a premium design, privacy camera shutter and long battery for meetings and travel."
Microsoft,Surface Book 3,Core i7,Intel,1.3 GHz,32GB,SSD,PixelSense,15 inch,NVIDIA GTX,3240x2160,Windows 11,1.9 kg,Detachable Screen,1 year,15 hours,180000,"The Microsoft Surface Book 3 is powered by an Intel Core i7 processor running at 1.3 GHz, paired with 32GB of RAM and SSD storage. It has a 15 inch PixelSense display with a resolution of 3240x2160 and uses NVIDIA GTX graphics. It runs Windows 11, weighs 1.9 kg, and offers Detachable Screen, a warranty of 1 year and around 15 hours of battery life. It is priced at Rs. 180,000. Suited to developers and technical professionals who need a detachable screen, 32GB RAM and a GPU for coding, testing and light ML work."
Lenovo,Yoga,Core i5,Intel,2.4 GHz,8GB,SSD,IPS,14 inch,Intel Iris Xe,1920x1080,Windows 11,1.4 kg,"Convertible, Touchscreen",1 year,10 hours,65000,"The Lenovo Yoga is powered by an Intel Core i5 processor running at 2.4 GHz, paired with 8GB of RAM and SSD storage. It has a 14 inch IPS display with a resolution of 1920x1080 and uses Intel Iris Xe graphics. It runs Windows 11, weighs 1.4 kg, and offers Convertible, Touchscreen, a warranty of 1 year and around 10 hours of battery life. It is priced at Rs. 65,000. A flexible 2-in-1 for general everyday use: streaming, browsing, video calls and light productivity at home or on the move."
ASUS,VivoBook,Core i3,Intel,2.1 GHz,8GB,SSD,LCD,15.6 inch,Intel UHD,1920x1080,Windows 11,1.8 kg,Fingerprint Sensor,1 year,6 hours,32000,"The ASUS VivoBook is powered by an Intel Core i3 processor running at 2.1 GHz, paired with 8GB of RAM and SSD storage. It has a 15.6 inch LCD display with a resolution of 1920x1080 and uses Intel UHD graphics. It runs Windows 11, weighs 1.8 kg, and offers Fingerprint Sensor, a warranty of 1 year and around 6 hours of battery life. It is priced at Rs. 32,000. An entry-level laptop for first-time buyers and students who need a dependable machine for everyday tasks."
"""

with open("laptop_data.csv", "w") as f:
    f.write(LAPTOP_CSV)

df = pd.read_csv("laptop_data.csv")
print(df.shape)
df[["Brand", "Model Name", "Price"]]
df

In [ ]:
df.iloc[1]

---
## MCQ 1 – Laptop category classification

Classify each laptop description into exactly one category: **gamer**, **business**, **general** or **programmer**.

**1.1** Write the system prompt for the classifier.

In [ ]:
category_system_prompt = """
You are a laptop classifier. Read the laptop description and assign it to exactly one category:
- gamer: built for gaming (dedicated NVIDIA GTX/RTX graphics, high-refresh display, RGB keyboard, cooling).
- business: for professionals, executives and corporate use (security, fingerprint, long battery life for travel/meetings, durability).
- programmer: for software developers, data scientists and ML work (high RAM, powerful CPU, coding, Docker, compiling).
- general: everyday use such as browsing, streaming, studying, documents and light office work.
Reply with ONLY the category name in lowercase (gamer, business, programmer or general). No punctuation or explanation.
"""

**1.2** Write a function that sends one description to the Chat Completions API and returns the category.

In [ ]:
VALID_CATEGORIES = {"gamer", "business", "general", "programmer"}

def get_laptop_category(description):
    messages = [
        {"role": "system", "content": category_system_prompt},
        {"role": "user", "content": description},
    ]
    category = get_chat_completion(messages).lower().strip(" .\n\"'")
    return category if category in VALID_CATEGORIES else "general"

get_laptop_category(df.iloc[1]["Description"])

**1.3** Apply the function to every description and store the result in a new column `Category`.

In [ ]:
df["Category"] = df["Description"].apply(get_laptop_category)
df[["Brand", "Model Name", "Category"]]

### Q1. What is the laptop category of the laptop at index 1 (MSI GL65)?

In [ ]:
df.iloc[1]['Category']

### Q2. How many gaming laptops did you get?

In [ ]:
print(df["Category"].value_counts())
print("Number of gaming laptops:", (df["Category"] == "gamer").sum())

---
## MCQ 2 – Extracting laptop tags as JSON

Extract these tags from each description and return them as a JSON object:
`Brand`, `Model Name`, `GPU processor`, `Display Resolution`, `Weight`, `Processor`, `Clock speed`, `Budget`.

`Budget` should be the laptop price in Indian number format (e.g. `85,000`, `2,00,000`).

**2.1** Write the system prompt for tag extraction.

In [ ]:
tags_system_prompt = """
You extract laptop tags from a product description.
Return ONLY a JSON object with exactly these keys:
"Brand", "Model Name", "GPU processor", "Display Resolution", "Weight", "Processor", "Clock speed", "Budget".
Use values exactly as stated in the description (e.g. Weight "2.9 kg", Clock speed "2.9 GHz",
Display Resolution "1920x1080", Processor like "Intel Core i7").
"Budget" is the price in Indian number format with commas, e.g. "85,000" or "2,00,000".
Do not include any text outside the JSON object.
"""

**2.2** Write a function that returns the tags as a Python dictionary.

In [ ]:
def get_laptop_tags(description):
    messages = [
        {"role": "system", "content": tags_system_prompt},
        {"role": "user", "content": description},
    ]
    reply = get_chat_completion(messages, json_mode=True)
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        return {"error": reply}

get_laptop_tags(df.iloc[1]["Description"])

**2.3** Apply the function to every description and store the result in a new column `laptop_tags`.

In [ ]:
df["laptop_tags"] = df["Description"].apply(get_laptop_tags)
df[["Brand", "Model Name", "laptop_tags"]]

### Q3. What are the laptop tags for index 10 (ASUS ROG Strix G)?

In [ ]:
print(json.dumps(df.iloc[10]["laptop_tags"], indent=2))

---
## MCQ 3 – Question answering with Wikipedia

Pipeline:
1. Ask the model to turn the user's question into a short Wikipedia search query.
2. Search Wikipedia (MediaWiki API) and fetch the text of the top pages.
3. Ask the model to answer the question **using only that Wikipedia text**.

In [ ]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
# Wikipedia requires a descriptive User-Agent header
WIKI_HEADERS = {"User-Agent": "NLP-Practice-Notebook/1.0 (educational use)"}

**3.1** Generate a Wikipedia search query from the user's question.

In [ ]:
def get_search_query(question):
    messages = [
        {"role": "system", "content": "Convert the user's question into a short Wikipedia search query (a few keywords or an article title). Reply with ONLY the query, nothing else."},
        {"role": "user", "content": question},
    ]
    return get_chat_completion(messages).strip('"')

get_search_query("Which country is hosting the FIFA world cup in 2026?")

**3.2** Search Wikipedia and return the plain text of the top results.

In [ ]:
def search_wikipedia(query, n_pages=3, max_chars=6000):
    # Step 1: find matching page titles
    search = requests.get(WIKI_API, headers=WIKI_HEADERS, params={
        "action": "query", "list": "search", "srsearch": query,
        "srlimit": n_pages, "format": "json",
    }).json()
    titles = [hit["title"] for hit in search["query"]["search"]]

    # Step 2: fetch the plain-text content of each page
    pages = []
    for title in titles:
        data = requests.get(WIKI_API, headers=WIKI_HEADERS, params={
            "action": "query", "prop": "extracts", "explaintext": 1,
            "redirects": 1, "titles": title, "format": "json",
        }).json()
        for page in data["query"]["pages"].values():
            text = page.get("extract", "")
            if text:
                pages.append(f"### {page['title']}\n{text[:max_chars]}")
    return titles, "\n\n".join(pages)

titles, context = search_wikipedia("2026 FIFA World Cup")
print(titles)
print(context[:500])

**3.3** Answer the question using only the Wikipedia context.

In [ ]:
def answer_with_wikipedia(question, verbose=True):
    query = get_search_query(question)
    titles, context = search_wikipedia(query)
    if verbose:
        print(f"Search query: {query}\nPages used:   {titles}\n")
    messages = [
        {"role": "system", "content": "Answer the question using ONLY the provided Wikipedia context. If the answer is not in the context, say that it was not found. Be concise."},
        {"role": "user", "content": f"Wikipedia context:\n{context}\n\nQuestion: {question}"},
    ]
    return get_chat_completion(messages)

### Q4. What is height of the tallest building built in the 18th century?

In [ ]:
print(answer_with_wikipedia("What is height of the tallest building built in the 18th century?"))

### Q5. Which country is hosting the FIFA world cup in 2026?

In [ ]:
print(answer_with_wikipedia("Which country is hosting the FIFA world cup in 2026?"))

### Q6. Which film won the best film award at Oscars 2024?

In [ ]:
print(answer_with_wikipedia("Which film won the best film award at Oscars 2024?"))